# Revisión de la preparación de datos

Este notebook revisa los archivos generados por DVC. No crea ni modifica los datos oficiales.

## P1 — Tipado

**Pregunta:** ¿La primera transformación conserva todas las filas y convierte únicamente las fechas?

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data/raw/cfpb_reclamos_narrativa.parquet"
TYPED_PATH = PROJECT_ROOT / "data/interim/typed.parquet"

assert RAW_PATH.exists(), f"No se encontró {RAW_PATH}"
assert TYPED_PATH.exists(), f"No se encontró {TYPED_PATH}; ejecuta dvc repro"

### 1. Filas, columnas y tamaño

In [3]:
raw = pq.ParquetFile(RAW_PATH)
typed = pq.ParquetFile(TYPED_PATH)

file_summary = pd.DataFrame({
    "archivo": ["original", "tipado"],
    "filas": [raw.metadata.num_rows, typed.metadata.num_rows],
    "columnas": [raw.metadata.num_columns, typed.metadata.num_columns],
    "tamaño GiB": [RAW_PATH.stat().st_size / 1024**3, TYPED_PATH.stat().st_size / 1024**3],
})
file_summary["filas"] = file_summary["filas"].map("{:,}".format)
file_summary["tamaño GiB"] = file_summary["tamaño GiB"].map("{:.2f}".format)
display(file_summary)

assert typed.metadata.num_rows == raw.metadata.num_rows == 3_837_184
assert typed.metadata.num_columns == raw.metadata.num_columns == 16

,archivo,filas,columnas,tamaño GiB
0,original,"3,837,184",16,1.51
1,tipado,"3,837,184",16,0.85


### 2. Tipos antes y después

In [4]:
raw_schema = raw.schema_arrow
typed_schema = typed.schema_arrow
schema_comparison = pd.DataFrame({
    "columna": raw_schema.names,
    "tipo original": [str(field.type) for field in raw_schema],
    "tipo tipado": [str(field.type) for field in typed_schema],
})
schema_comparison["cambió"] = schema_comparison["tipo original"] != schema_comparison["tipo tipado"]
display(schema_comparison)

changed_columns = schema_comparison.loc[schema_comparison["cambió"], "columna"].tolist()
assert changed_columns == ["Date received", "Date sent to company"]
assert typed_schema.field("Date received").type == pa.date32()
assert typed_schema.field("Date sent to company").type == pa.date32()

,columna,tipo original,tipo tipado,cambió
0,Date received,large_string,date32[day],True
1,Product,large_string,large_string,False
2,Sub-product,large_string,large_string,False
3,Issue,large_string,large_string,False
4,Sub-issue,large_string,large_string,False
5,Consumer complaint narrative,large_string,large_string,False
6,Company public response,large_string,large_string,False
7,Company,large_string,large_string,False
8,State,large_string,large_string,False
9,ZIP code,large_string,large_string,False


### 3. Calidad de las fechas

In [5]:
typed_dates = pq.read_table(TYPED_PATH, columns=["Date received", "Date sent to company"])
date_summary = []
for column in typed_dates.column_names:
    values = typed_dates[column]
    date_summary.append({
        "columna": column,
        "mínimo": pc.min(values).as_py(),
        "máximo": pc.max(values).as_py(),
        "nulos": values.null_count,
    })
display(pd.DataFrame(date_summary))
assert all(row["nulos"] == 0 for row in date_summary)

,columna,mínimo,máximo,nulos
0,Date received,2015-03-19,2026-07-27,0
1,Date sent to company,2015-03-19,2026-08-13,0


### 4. Identificadores

In [6]:
complaint_ids = pq.read_table(TYPED_PATH, columns=["Complaint ID"])["Complaint ID"]
unique_ids = pc.count_distinct(complaint_ids).as_py()
display(pd.DataFrame({
    "medida": ["filas", "IDs únicos", "IDs nulos"],
    "valor": [len(complaint_ids), unique_ids, complaint_ids.null_count],
}))
assert unique_ids == len(complaint_ids) == 3_837_184
assert complaint_ids.null_count == 0

,medida,valor
0,filas,3837184
1,IDs únicos,3837184
2,IDs nulos,0


## Resultado de P1

- Se conservan las 3,837,184 filas y las 16 columnas.
- Solo cambian `Date received` y `Date sent to company`: pasan de texto a fecha.
- No se crean fechas nulas.
- Los 3,837,184 identificadores continúan presentes y son únicos.
- Ninguna categoría, narrativa u objetivo se modifica en esta etapa.

## P2 — Normalización de narrativas

**Normalizar** significa aplicar reglas fijas para que textos equivalentes puedan compararse. La versión `text_normalizer_v1` aplica Unicode NFKC, minúsculas con `casefold`, reemplazo de palabras formadas por dos o más `x` por `<redacted>`, unión de espacios y retiro de espacios al inicio y final.

Un **hash SHA-256** es un identificador estable calculado desde el texto normalizado. El mismo texto produce el mismo identificador; esto permite formar grupos sin usar la narrativa completa como llave.

In [7]:
from itertools import zip_longest

from src.data.normalize_text import (
    HASH_COLUMN,
    NARRATIVE_COLUMN,
    NORMALIZED_COLUMN,
    NORMALIZER_VERSION,
    hash_text,
    normalize_text,
)

NORMALIZED_PATH = PROJECT_ROOT / "data/interim/normalized.parquet"
assert NORMALIZED_PATH.exists(), f"No se encontró {NORMALIZED_PATH}; ejecuta dvc repro"

### 1. Filas, columnas y versión

In [8]:
normalized_file = pq.ParquetFile(NORMALIZED_PATH)
normalizer_version = normalized_file.schema_arrow.metadata[b"text_normalizer"].decode("utf-8")
p2_file_summary = pd.DataFrame({
    "archivo": ["tipado", "normalizado"],
    "filas": [typed.metadata.num_rows, normalized_file.metadata.num_rows],
    "columnas": [typed.metadata.num_columns, normalized_file.metadata.num_columns],
    "tamaño GiB": [TYPED_PATH.stat().st_size / 1024**3, NORMALIZED_PATH.stat().st_size / 1024**3],
})
p2_file_summary["filas"] = p2_file_summary["filas"].map("{:,}".format)
p2_file_summary["tamaño GiB"] = p2_file_summary["tamaño GiB"].map("{:.2f}".format)
display(p2_file_summary)
print(f"Versión: {normalizer_version}")

assert normalized_file.metadata.num_rows == typed.metadata.num_rows == 3_837_184
assert normalized_file.schema_arrow.names == typed.schema_arrow.names + [NORMALIZED_COLUMN, HASH_COLUMN]
assert normalizer_version == NORMALIZER_VERSION

,archivo,filas,columnas,tamaño GiB
0,tipado,"3,837,184",16,0.85
1,normalizado,"3,837,184",18,1.70


Versión: text_normalizer_v1


### 2. Conservación de la narrativa original

La comparación se realiza por lotes para revisar todas las filas sin cargar ambas tablas completas en memoria.

In [9]:
typed_batches = typed.iter_batches(batch_size=50_000, columns=[NARRATIVE_COLUMN])
normalized_batches = normalized_file.iter_batches(batch_size=50_000, columns=[NARRATIVE_COLUMN])
original_is_unchanged = True
for before, after in zip_longest(typed_batches, normalized_batches):
    if before is None or after is None or not before.equals(after):
        original_is_unchanged = False
        break

display(pd.DataFrame({"comprobación": ["narrativa original sin cambios"], "resultado": [original_is_unchanged]}))
assert original_is_unchanged

,comprobación,resultado
0,narrativa original sin cambios,True


### 3. Calidad de las columnas nuevas

In [10]:
def parquet_null_count(parquet_file, column_name):
    column_index = parquet_file.schema_arrow.get_field_index(column_name)
    return sum(
        parquet_file.metadata.row_group(index).column(column_index).statistics.null_count
        for index in range(parquet_file.metadata.num_row_groups)
    )

sample = next(normalized_file.iter_batches(batch_size=1_000, columns=[NORMALIZED_COLUMN, HASH_COLUMN]))
sample_normalized = sample.column(0).to_pylist()
sample_hashes = sample.column(1).to_pylist()
sample_hashes_are_correct = all(hash_text(text) == value for text, value in zip(sample_normalized, sample_hashes))
sample_is_idempotent = all(normalize_text(text) == text for text in sample_normalized)
normalized_nulls = parquet_null_count(normalized_file, NORMALIZED_COLUMN)
hash_nulls = parquet_null_count(normalized_file, HASH_COLUMN)

display(pd.DataFrame({
    "comprobación": ["textos normalizados nulos", "hashes nulos", "hash correcto en muestra", "regla estable en muestra"],
    "resultado": [normalized_nulls, hash_nulls, sample_hashes_are_correct, sample_is_idempotent],
}))
assert normalized_nulls == hash_nulls == 0
assert sample_hashes_are_correct
assert sample_is_idempotent

,comprobación,resultado
0,textos normalizados nulos,0
1,hashes nulos,0
2,hash correcto en muestra,True
3,regla estable en muestra,True


### 4. Grupos de texto normalizado

Un **grupo repetido** contiene al menos dos filas con el mismo hash y, por tanto, con el mismo texto después de aplicar las reglas aprobadas.

In [11]:
hashes = pq.read_table(NORMALIZED_PATH, columns=[HASH_COLUMN])[HASH_COLUMN].combine_chunks()
hash_counts = pc.value_counts(hashes)
counts = hash_counts.field("counts")
repeated_mask = pc.greater(counts, 1)
unique_hashes = len(counts)
repeated_groups = pc.sum(pc.cast(repeated_mask, pa.int64())).as_py()
rows_in_repeated_groups = pc.sum(pc.filter(counts, repeated_mask)).as_py()
repeated_percentage = rows_in_repeated_groups / len(hashes) * 100

display(pd.DataFrame({
    "medida": ["textos normalizados únicos", "grupos repetidos", "filas en grupos repetidos", "porcentaje de filas repetidas"],
    "valor": [f"{unique_hashes:,}", f"{repeated_groups:,}", f"{rows_in_repeated_groups:,}", f"{repeated_percentage:.2f}%"],
}))
assert unique_hashes == 2_476_614
assert repeated_groups == 281_943
assert rows_in_repeated_groups == 1_642_513

,medida,valor
0,textos normalizados únicos,"2,476,614"
1,grupos repetidos,"281,943"
2,filas en grupos repetidos,"1,642,513"
3,porcentaje de filas repetidas,42.81%


## Resultado de P2

- Se conservan las 3,837,184 filas y las 16 columnas originales.
- Se agregan el texto normalizado y su hash SHA-256; ninguna de estas columnas tiene valores ausentes.
- La regla aplicada queda identificada como `text_normalizer_v1`.
- Existen 2,476,614 textos normalizados distintos.
- 1,642,513 filas, 42.81%, comparten texto normalizado con otra fila.
- Compartir texto no demuestra que dos reclamos sean el mismo evento. El hash se usará para evitar que un mismo patrón textual quede dividido entre aprendizaje y evaluación.